### CO2 Rating Predcition Using Different ML algorithms and Comparing them

--- Data Overview & Preprocessing ---

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,log_loss
from sklearn.preprocessing import StandardScaler

df=pd.read_csv("fuel_data.csv")

# data overview
print("first five rows:\n",df.head(5))
print("\nShape of the data\n:",df.shape)
print("\ncolumns of data:\n",df.columns)
print("\nnull values:\n",df.isnull().sum())
df.info()

# DATA PREPROCESSING

fuel_type_dict = {
    "X": "Regular Gasoline",
    "Z": "Premium Gasoline",
    "D": "Diesel",
    "E": "Ethanol (E85)"
}
df["Fuel type"]=df["Fuel type"].replace(fuel_type_dict)

df_encoded = pd.get_dummies(df, columns=["Vehicle class","Transmission","Fuel type"], drop_first=True)

first five rows:
    Model year   Make              Model                    Vehicle class  \
0        2025  Acura     Integra A-SPEC                        Full-size   
1        2025  Acura     Integra A-SPEC                        Full-size   
2        2025  Acura     Integra Type S                        Full-size   
3        2025  Acura         MDX SH-AWD     Sport utility vehicle: Small   
4        2025  Acura  MDX SH-AWD Type S  Sport utility vehicle: Standard   

   Engine size (L)  Cylinders Transmission Fuel type  City (L/100 km)  \
0              1.5          4          AV7         Z              8.1   
1              1.5          4           M6         Z              8.9   
2              2.0          4           M6         Z             11.1   
3              3.5          6         AS10         Z             12.6   
4              3.0          6         AS10         Z             13.8   

   Highway (L/100 km)  Combined (L/100 km)  Combined (mpg)  \
0                 6.5   

# CLASSIFICATION MODELS

### Logistic Regression for CO2 rating


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

X=df_encoded.drop(columns=["Make","Model","Smog rating","CO2 emissions (g/km)","CO2 rating","City (L/100 km)",
                           "Highway (L/100 km)","Combined (L/100 km)","Combined (mpg)"],axis=1)
y = (df_encoded["CO2 rating"] >= 5).astype(int)

pipe=Pipeline([('scaler',StandardScaler()),
               ('model',LogisticRegression(max_iter=1000))
               ])
param_grid={'model__C': [0.01, 0.1, 1, 10],
            'model__solver': ['lbfgs']
            }

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

grid=GridSearchCV(pipe,param_grid,cv=5,scoring='accuracy')
grid.fit(X_train,y_train)
best_model=grid.best_estimator_
y_pred=best_model.predict(X_test)

print("Best Parameter:",grid.best_params_)
print("\nAccuracy:\n",accuracy_score(y_test,y_pred))
print("\nConfusion matrix:\n",confusion_matrix(y_test,y_pred))
print("\n=== Classification Report ===\n",classification_report(y_test,y_pred))

Best Parameter: {'model__C': 10, 'model__solver': 'lbfgs'}

Accuracy:
 0.8769230769230769

Confusion matrix:
 [[50  8]
 [ 8 64]]

=== Classification Report ===
               precision    recall  f1-score   support

           0       0.86      0.86      0.86        58
           1       0.89      0.89      0.89        72

    accuracy                           0.88       130
   macro avg       0.88      0.88      0.88       130
weighted avg       0.88      0.88      0.88       130



### KNN Classification for CO2 Rating

In [15]:
from sklearn.neighbors import KNeighborsClassifier

pipe=Pipeline([('scaler',StandardScaler()),
               ('model',KNeighborsClassifier())
               ])

param_grid={'model__n_neighbors':[3,5,7,9],
            'model__weights':['uniform','distance'],
            'model__p':[1,2]
            }

grid=GridSearchCV(pipe,param_grid,cv=5,scoring='accuracy')
grid.fit(X_train,y_train)
best_model=grid.best_estimator_
y_pred=best_model.predict(X_test)

print("Best Parameter:",grid.best_params_)
print("\nAccuracy:\n",accuracy_score(y_test,y_pred))
print("\nConfusion matrix:\n",confusion_matrix(y_test,y_pred))
print("\n=== Classification Report ===\n",classification_report(y_test,y_pred))

Best Parameter: {'model__n_neighbors': 9, 'model__p': 1, 'model__weights': 'distance'}

Accuracy:
 0.823076923076923

Confusion matrix:
 [[50  8]
 [15 57]]

=== Classification Report ===
               precision    recall  f1-score   support

           0       0.77      0.86      0.81        58
           1       0.88      0.79      0.83        72

    accuracy                           0.82       130
   macro avg       0.82      0.83      0.82       130
weighted avg       0.83      0.82      0.82       130



### Decision Tree for CO2 Rating

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model=DecisionTreeClassifier()
    
param_grid={'max_depth':[None,5,10,15],
            'min_samples_leaf':[1,2,4],
            'min_samples_split':[2,5,10],
            'criterion':['gini','entropy'],
            'random_state':[42]
            }

grid_dt=GridSearchCV(model,param_grid,cv=5,scoring='accuracy')
grid_dt.fit(X_train,y_train)
best_model=grid_dt.best_estimator_
y_pred=best_model.predict(X_test)

print("Best Parameter:",grid_dt.best_params_)
print("\nAccuracy:\n",accuracy_score(y_test,y_pred))
print("\n=== Classification Report ===\n",classification_report(y_test,y_pred))
print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred))

Best Parameter: {'criterion': 'entropy', 'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 2, 'random_state': 42}

Accuracy:
 0.8923076923076924

=== Classification Report ===
               precision    recall  f1-score   support

           0       0.89      0.86      0.88        58
           1       0.89      0.92      0.90        72

    accuracy                           0.89       130
   macro avg       0.89      0.89      0.89       130
weighted avg       0.89      0.89      0.89       130

Confusion Matrix:
 [[50  8]
 [ 6 66]]


### Support Vector Machine

In [19]:
from sklearn.svm import SVC

pipe=Pipeline([('scaler',StandardScaler()),
               ('model',SVC())
               ])

param_grid={'model__C':[0.1,1,10],
            'model__kernel':['rbf','linear'],
            'model__gamma':['scale','auto']
            }

grid_svc=GridSearchCV(pipe,param_grid,cv=5,scoring='accuracy')
grid_svc.fit(X_train,y_train)
best_model=grid_svc.best_estimator_
y_pred_svc=best_model.predict(X_test)

print("Best Parameter:",grid_svc.best_params_)

print("Predicted Values are:\n",y_pred_svc)
print("\nAccuracy:\n",accuracy_score(y_test,y_pred_svc))
print("\nConfusion Matrix:\n",confusion_matrix(y_test,y_pred_svc))
print("\nClassification Report:\n",classification_report(y_test,y_pred_svc))

Best Parameter: {'model__C': 10, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Predicted Values are:
 [1 0 1 1 0 0 1 0 1 0 1 1 0 0 1 0 0 1 0 1 1 1 1 0 0 0 1 0 1 1 1 0 0 0 0 1 1
 1 1 0 0 1 0 0 0 1 0 1 1 0 1 1 0 1 1 0 1 1 0 0 0 0 0 1 1 0 1 0 0 1 0 1 1 1
 0 0 1 1 1 1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0
 1 0 0 0 1 0 1 1 1 0 0 1 0 0 1 0 0 0 1]

Accuracy:
 0.8615384615384616

Confusion Matrix:
 [[54  4]
 [14 58]]

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.93      0.86        58
           1       0.94      0.81      0.87        72

    accuracy                           0.86       130
   macro avg       0.86      0.87      0.86       130
weighted avg       0.87      0.86      0.86       130



### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score

model_nb=GaussianNB()

scores = cross_val_score(model_nb, X_train, y_train, cv=5, scoring='accuracy')
print("Naive Bayes CV Accuracy:", scores.mean())

model_nb.fit(X_train,y_train)
y_pred_nb=model_nb.predict(X_test)

print("\nPredicted Values:\n",y_pred_nb)
print("Accuracy:",accuracy_score(y_test,y_pred_nb))
print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred_nb))
print("Classification Report:\n",classification_report(y_test,y_pred_nb))

Naive Bayes CV Accuracy: 0.6538461538461539

Predicted Values:
 [0 0 1 1 0 0 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 0 1 0
 1 1 0 0 1 0 0 0 0 0 1 1 0 1 0 0 1 0 0 1 1 0 0 0 0 0 0 1 0 1 0 0 1 0 1 0 1
 0 0 0 1 0 1 0 0 0 1 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1]
Accuracy: 0.676923076923077
Confusion Matrix:
 [[57  1]
 [41 31]]
Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.98      0.73        58
           1       0.97      0.43      0.60        72

    accuracy                           0.68       130
   macro avg       0.78      0.71      0.66       130
weighted avg       0.80      0.68      0.66       130



### Random Forest Classifier

In [22]:
from sklearn.ensemble import RandomForestClassifier

rf_model=RandomForestClassifier()

param_grid={
    'n_estimators':[100,200],
    'max_depth':[None,10,20],
    'min_samples_split':[2,3,5],
    'min_samples_leaf':[1,2,3]
}
grid=GridSearchCV(rf_model,param_grid,cv=5)
grid.fit(X_train,y_train)
best_rf = grid.best_estimator_

y_pred_rf = best_rf.predict(X_test)
print(grid.best_params_)
print("The co2 emission prediction is:",y_pred_rf)

print("Accuracy:",accuracy_score(y_test,y_pred_rf))
print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred_rf))
print("Classification Report:\n",classification_report(y_test,y_pred_rf))

{'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
The co2 emission prediction is: [1 0 1 1 0 0 1 0 1 0 1 1 1 0 1 1 0 1 0 1 1 1 1 1 0 1 1 0 1 1 1 0 0 0 0 1 1
 1 1 0 0 1 0 0 0 1 0 1 1 1 1 1 0 1 1 0 1 1 0 0 0 0 0 1 1 0 1 0 0 1 0 1 1 1
 0 0 1 1 1 1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 0 0 1 0 0 0 0 0 1 0 0 1 1 0 1 0 0
 1 0 0 0 1 0 1 1 1 0 0 1 0 0 1 0 0 0 1]
Accuracy: 0.8538461538461538
Confusion Matrix:
 [[50  8]
 [11 61]]
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.86      0.84        58
           1       0.88      0.85      0.87        72

    accuracy                           0.85       130
   macro avg       0.85      0.85      0.85       130
weighted avg       0.86      0.85      0.85       130

